In [1]:
# Imports
import hashlib
from pathlib import Path

# Defining Paths
TRAIN_DIR = Path("../dataset/train")
TEST_DIR = Path("../dataset/test")

In [5]:
# --- 1. Exact Duplicate Check ---

# MD5 Calculator
def calculate_md5(file_path):
    hasher = hashlib.md5()
    with open(file_path, "rb") as f:
        buf = f.read()
        hasher.update(buf)
    return hasher.hexdigest()

# Internal Duplicate Finder
def find_internal_duplicates(file_path):
    hashes = {}
    duplicates = []

    for img_path in file_path.glob("**/*.jpg"):
        file_hash = calculate_md5(img_path)
        if file_hash in hashes:
            duplicates.append((img_path, hashes[file_hash]))
        else:
            hashes[file_hash] = img_path
    return duplicates,hashes

In [4]:
# --- 2. Train and Test Duplicate Control ---
print("Scanning train dataset for internal duplicates...")
train_duplicates, train_hashes = find_internal_duplicates(TRAIN_DIR)

print("Scanning test dataset for internal duplicates...")
test_duplicates, test_hashes = find_internal_duplicates(TEST_DIR)

# --- 2.1. Exact Duplicate Analysis Report ---
print("\n" + "=" * 45)
print("EXACT DUPLICATE ANALYSIS REPORT")
print("=" * 45)
print(f"Train Set Internal Duplicates : {len(train_duplicates)}")
print(f"Test Set Internal Duplicates  : {len(test_duplicates)}")
print(f"Total Internal Duplicates     : {len(train_duplicates) + len(test_duplicates)}")
print("=" * 45)

if len(train_duplicates) > 0 or len(test_duplicates) > 0:
    print("⚠️ Warning: Internal duplicates detected! You may want to review and clean them.")
else:
    print("✨ Clean! No internal duplicates found in train or test sets.")

Scanning train dataset for internal duplicates...
Scanning test dataset for internal duplicates...

EXACT DUPLICATE ANALYSIS REPORT
Train Set Internal Duplicates : 32
Test Set Internal Duplicates  : 0
Total Internal Duplicates     : 32
⚠️ Warning: Internal duplicates detected! You may want to review and clean them.


In [7]:
# --- 3. Cross-Split Leakage Control  ---
print("Checking train-test leakage (Cross-split duplicates)...")
cross_leakage = []

for img_path in TEST_DIR.glob("**/*.jpg"):
    file_hash = calculate_md5(img_path)
    if file_hash in train_hashes:
        cross_leakage.append({
            "test_file": img_path,
            "matching_train_file": train_hashes[file_hash]
        })

# Results
print("\n" + "="*45)
print("DUPLICATE & LEAKAGE ANALYSIS REPORT")
print("="*45)
print(f"Train-Test Sızıntı (Leakage)  : {len(cross_leakage)}")
print("="*45)

Checking train-test leakage (Cross-split duplicates)...

DUPLICATE & LEAKAGE ANALYSIS REPORT
Train-Test Sızıntı (Leakage)  : 3000


In [ ]:
# --- 4. Fix Data Leakage: Remove overlapping images from Test Set ---
removed_leakage_count = 0

print("Cleaning data leakage from the test set...")
for item in cross_leakage:
    test_file_path = item["test_file"]
    try:
        if test_file_path.exists():
            test_file_path.unlink()
            removed_leakage_count += 1
    except Exception as e:
        print(f"Error removing file {test_file_path}: {e}")

print(f"\nSuccessfully removed {removed_leakage_count} leaked images from the test set.")